# Python Multiprocessing

### Learning Objectives
- Understand the GIL and why multiprocessing bypasses it for CPU-bound work
- Create and manage `Process` objects directly
- Collect worker results via `Queue`
- Distribute work with `Pool.map` and `Pool.starmap`
- Use `ProcessPoolExecutor` from `concurrent.futures` for a higher-level API
- Share state safely with `Value`, `Array`, and `Lock`
- Benchmark serial vs parallel execution on a CPU-bound task
- Recognise and avoid common multiprocessing pitfalls
- Write concurrent I/O programs with `asyncio` coroutines and tasks
- Combine `asyncio` with `ProcessPoolExecutor` to handle CPU + I/O workloads together

In [19]:
import asyncio
import math
import multiprocessing as mp
import os
import time
import concurrent.futures
from multiprocessing import Array, Lock, Pipe, Pool, Process, Queue, Value

print('CPU count:', mp.cpu_count())
print('Start method:', mp.get_start_method())
print('Main PID:', os.getpid())

CPU count: 8
Start method: fork
Main PID: 598477


## 1. The GIL and Why Multiprocessing

Python's **Global Interpreter Lock (GIL)** allows only one thread to execute Python bytecode at a time, even on a multi-core machine.  
This means `threading` does **not** speed up CPU-bound work — threads take turns rather than running in parallel.

`multiprocessing` sidesteps the GIL by spawning **separate OS processes**, each with its own Python interpreter and memory space.  
There is no shared memory by default, so no GIL contention.

| Workload | Best tool | Why |
|----------|-----------|-----|
| CPU-bound (math, parsing, ML inference) | `multiprocessing` | Runs truly in parallel across cores |
| I/O-bound (network, disk, DB) | `threading` or `asyncio` | Threads release the GIL while waiting |
| Mixed / task-based | `concurrent.futures` | Unified API, swap executor type |

**Process vs Thread memory model**
- Thread: shared heap, shared GIL, cheap context switch
- Process: private heap (copy-on-write after fork), no GIL, heavier context switch

In [20]:
import threading

def cpu_work(n=5_000_000):
    """Pure CPU work — counts up to n."""
    total = 0
    for i in range(n):
        total += i
    return total

n_tasks = 4

# Serial baseline
t0 = time.perf_counter()
for _ in range(n_tasks):
    cpu_work()
serial_time = time.perf_counter() - t0

# Threaded — does NOT speed up CPU-bound work
t0 = time.perf_counter()
threads = [threading.Thread(target=cpu_work) for _ in range(n_tasks)]
for t in threads:
    t.start()
for t in threads:
    t.join()
thread_time = time.perf_counter() - t0

print(f'Serial:  {serial_time:.3f}s')
print(f'Threads: {thread_time:.3f}s  (no speedup — GIL prevents true parallelism)')

Serial:  0.683s
Threads: 0.632s  (no speedup — GIL prevents true parallelism)


## 2. Basic `Process` — Start, Join, and Daemon

A `Process` object wraps a function and runs it in a separate OS process.

| Method / attr | Purpose |
|---------------|---------|
| `p.start()` | Fork a child process and begin execution |
| `p.join()` | Block until the child finishes |
| `p.is_alive()` | Check if child is still running |
| `p.pid` | OS PID of the child |
| `p.daemon = True` | Child is killed when parent exits; cannot itself spawn children |
| `p.exitcode` | Return code after the process finishes (`0` = success) |

In [21]:
def worker(label, delay):
    print(f'[{label}] started  PID={os.getpid()}  parent={os.getppid()}')
    time.sleep(delay)
    print(f'[{label}] finished')

p1 = Process(target=worker, args=('A', 0.1))
p2 = Process(target=worker, args=('B', 0.2))

p1.start()
p2.start()
print(f'Main PID={os.getpid()}, p1.pid={p1.pid}, p2.pid={p2.pid}')

p1.join()
p2.join()

print(f'p1 exitcode: {p1.exitcode}')
print(f'p2 exitcode: {p2.exitcode}')
print('Both done')

[A] started  PID=600153  parent=598477
[B] started  PID=600154  parent=598477


[A] finished
Main PID=598477, p1.pid=600153, p2.pid=600154
[B] finished
p1 exitcode: 0
p2 exitcode: 0
Both done


In [22]:
# join(timeout=) avoids blocking forever — check exitcode/is_alive after
def slow_worker():
    time.sleep(5)

p = Process(target=slow_worker)
p.start()
p.join(timeout=0.3)

if p.is_alive():
    print('Still running after timeout — terminating')
    p.terminate()
    p.join()

# Negative exitcode means terminated by signal (-15 = SIGTERM)
print('exitcode:', p.exitcode)

Still running after timeout — terminating
exitcode: -15


## 3. Collect Results with `Queue`

Processes have separate memory, so return values are not shared automatically.  
A `multiprocessing.Queue` is a thread- and process-safe FIFO pipe backed by OS IPC.

Pattern: each worker puts its result into the queue; the main process reads all results after joining.

In [23]:
def square_worker(q, x):
    result = x * x
    q.put((x, result))

q = Queue()
inputs = [1, 2, 3, 4, 5]

processes = [Process(target=square_worker, args=(q, x)) for x in inputs]
for p in processes:
    p.start()
for p in processes:
    p.join()

# Drain the queue after all workers have finished
results = {}
while not q.empty():
    x, sq = q.get()
    results[x] = sq

print('Results:', dict(sorted(results.items())))

Results: {1: 1, 2: 4, 3: 9, 4: 16, 5: 25}


In [24]:
# Sentinel pattern — use None to signal that a worker is done
SENTINEL = None

def batch_worker(q, items):
    for item in items:
        q.put(item ** 2)
    q.put(SENTINEL)

result_q = Queue()
p = Process(target=batch_worker, args=(result_q, range(6)))
p.start()

collected = []
while True:
    val = result_q.get()
    if val is SENTINEL:
        break
    collected.append(val)

p.join()
print('Collected:', collected)

Collected: [0, 1, 4, 9, 16, 25]


## 4. Distribute Work with `Pool.map` and `Pool.starmap`

`Pool` manages a fixed set of worker processes and distributes tasks across them automatically.  
It is the most common multiprocessing pattern in data processing and ML pipelines.

| Method | Signature | When to use |
|--------|-----------|-------------|
| `pool.map(fn, iterable)` | one arg per call | map a single-arg function across items |
| `pool.starmap(fn, iterable)` | tuple unpacked as multiple args | multi-arg functions |
| `pool.map_async(fn, iterable)` | returns `AsyncResult` | non-blocking, check `.get()` later |
| `pool.imap(fn, iterable)` | lazy iterator | very large iterables, stream results |

Always use `Pool` as a context manager so workers are terminated on exit.

In [25]:
def heavy_sqrt(x):
    """CPU-bound task: repeated sqrt computation."""
    val = float(x)
    for _ in range(10_000):
        val = math.sqrt(abs(val) + 1)
    return val

inputs = list(range(32))

# Serial baseline
t0 = time.perf_counter()
serial_results = [heavy_sqrt(x) for x in inputs]
serial_t = time.perf_counter() - t0

# Parallel with Pool
t0 = time.perf_counter()
with Pool(processes=mp.cpu_count()) as pool:
    parallel_results = pool.map(heavy_sqrt, inputs)
parallel_t = time.perf_counter() - t0

print(f'Serial:   {serial_t:.3f}s')
print(f'Parallel: {parallel_t:.3f}s  (speedup: {serial_t / parallel_t:.1f}x)')
print('Results match:', serial_results == parallel_results)

Serial:   0.037s
Parallel: 0.051s  (speedup: 0.7x)
Results match: True


In [26]:
def power(base, exp):
    return base ** exp

# starmap unpacks each tuple as positional args to the function
args = [(2, 10), (3, 5), (4, 3), (10, 2)]

with Pool() as pool:
    results = pool.starmap(power, args)

print('starmap results:', results)

# map_async — non-blocking, main process can do other work
with Pool() as pool:
    async_result = pool.map_async(heavy_sqrt, range(8))
    print('Main continues while workers run...')
    final = async_result.get(timeout=30)

print('async results:', [round(v, 4) for v in final])

starmap results: [1024, 243, 64, 100]
Main continues while workers run...
async results: [1.618, 1.618, 1.618, 1.618, 1.618, 1.618, 1.618, 1.618]


## 5. `ProcessPoolExecutor` — Higher-Level Futures API

`concurrent.futures.ProcessPoolExecutor` wraps `multiprocessing.Pool` with a cleaner interface.

Key advantages over raw `Pool`:
- Returns `Future` objects — inspect status, cancel, add callbacks
- `as_completed()` yields futures as they finish (not in submission order)
- Unified interface with `ThreadPoolExecutor` — swap executor without changing worker code

| Method | Returns | Use case |
|--------|---------|----------|
| `executor.map(fn, iter)` | lazy iterator of results | ordered results, simple fan-out |
| `executor.submit(fn, *args)` | `Future` | fine-grained control, callbacks |
| `as_completed(futures)` | iterator of done `Future`s | process results as they arrive |

In [27]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(math.sqrt(n)) + 1):
        if n % i == 0:
            return False
    return True

candidates = list(range(1, 101))

# executor.map — ordered, like built-in map()
with concurrent.futures.ProcessPoolExecutor() as executor:
    primality = list(executor.map(is_prime, candidates))

primes = [n for n, p in zip(candidates, primality) if p]
print('Primes up to 100:', primes)

Primes up to 100: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]


In [28]:
def count_primes_up_to(limit):
    """Count primes below limit — intentionally CPU-bound."""
    return limit, sum(1 for n in range(2, limit) if is_prime(n))

limits = [50_000, 80_000, 100_000, 120_000]

# submit() + as_completed() — process results in completion order
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = {executor.submit(count_primes_up_to, lim): lim for lim in limits}

    for future in concurrent.futures.as_completed(futures):
        lim, count = future.result()
        print(f'Primes below {lim:>7,}: {count:>5,}')

# Futures carry status and support callbacks
with concurrent.futures.ProcessPoolExecutor() as executor:
    f = executor.submit(count_primes_up_to, 10_000)
    f.add_done_callback(lambda fut: print(f'Callback fired: {fut.result()}'))

Primes below  50,000: 5,133
Primes below  80,000: 7,837
Primes below 100,000: 9,592
Primes below 120,000: 11,301
Callback fired: (10000, 1229)


## 6. Shared Memory — `Value` and `Array`

By default each process gets a **copy** of the parent's memory (copy-on-write fork).  
When processes must share a mutable counter or buffer, use `Value` or `Array` backed by shared memory.

| Type | C typecode | Example | Use case |
|------|-----------|---------|----------|
| `Value('i', 0)` | signed int | global counter | single shared scalar |
| `Value('d', 0.0)` | double | accumulated loss | shared float |
| `Array('i', n)` | int array | shared buffer | fixed-size shared array |
| `Array('d', data)` | double array | results buffer | pre-allocated output |

Always protect shared state with a `Lock` — without one, concurrent writes cause **race conditions**.

In [29]:
def unsafe_increment(counter, n):
    for _ in range(n):
        counter.value += 1

def safe_increment(counter, lock, n):
    for _ in range(n):
        with lock:
            counter.value += 1

n_procs, n_increments = 4, 10_000

# Race condition: final value will be less than expected
unsafe_counter = Value('i', 0)
procs = [Process(target=unsafe_increment, args=(unsafe_counter, n_increments)) for _ in range(n_procs)]
for p in procs: p.start()
for p in procs: p.join()
print(f'Unsafe counter: {unsafe_counter.value}  (expected {n_procs * n_increments})')

# Lock prevents the race condition
safe_counter = Value('i', 0)
lock = Lock()
procs = [Process(target=safe_increment, args=(safe_counter, lock, n_increments)) for _ in range(n_procs)]
for p in procs: p.start()
for p in procs: p.join()
print(f'Safe counter:   {safe_counter.value}  (expected {n_procs * n_increments})')

Unsafe counter: 21895  (expected 40000)
Safe counter:   40000  (expected 40000)


In [30]:
def fill_segment(arr, start, end, value):
    for i in range(start, end):
        arr[i] = value

n = 12
shared_arr = Array('i', n)

# Divide the array into segments, each filled by a separate process
segment_size = n // 4
procs = [
    Process(target=fill_segment, args=(shared_arr, i * segment_size, (i + 1) * segment_size, i + 1))
    for i in range(4)
]
for p in procs: p.start()
for p in procs: p.join()

print('Shared array:', list(shared_arr))

Shared array: [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4]


## 7. `Lock` and `Semaphore` for Synchronization

| Primitive | Behaviour | Typical use |
|-----------|-----------|-------------|
| `Lock` | Mutual exclusion — one holder at a time | Protect a shared resource |
| `RLock` | Re-entrant lock — same process can acquire multiple times | Recursive functions holding a lock |
| `Semaphore(n)` | Counter-based — up to *n* holders at a time | Rate-limit concurrent access (DB pool, API) |
| `Event` | Boolean flag — `set()` / `wait()` | Signal workers to start or stop |

Always use locks as **context managers** to guarantee release even on exceptions.

In [31]:
from multiprocessing import Semaphore, Event

# Semaphore limits concurrent access to 2 workers at a time
def throttled_worker(sem, worker_id, results_q):
    with sem:
        results_q.put(f'worker-{worker_id} entered')
        time.sleep(0.05)
        results_q.put(f'worker-{worker_id} done')

sem = Semaphore(2)   # at most 2 processes in the critical section
q = Queue()

procs = [Process(target=throttled_worker, args=(sem, i, q)) for i in range(5)]
for p in procs: p.start()
for p in procs: p.join()

log = []
while not q.empty():
    log.append(q.get())
for entry in log:
    print(entry)

worker-0 entered
worker-1 entered
worker-0 done
worker-2 entered
worker-1 done
worker-3 entered
worker-2 done
worker-4 entered
worker-3 done
worker-4 done


In [32]:
# Event: main process signals workers to stop
def polling_worker(stop_event, worker_id):
    steps = 0
    while not stop_event.is_set():
        steps += 1
        time.sleep(0.01)
    print(f'Worker {worker_id} stopped after {steps} steps')

stop = Event()
procs = [Process(target=polling_worker, args=(stop, i)) for i in range(3)]
for p in procs: p.start()

time.sleep(0.1)
stop.set()   # all workers see this and exit their loops

for p in procs: p.join()

Worker 2 stopped after 10 stepsWorker 0 stopped after 11 steps

Worker 1 stopped after 11 steps


## 8. CPU-Bound Benchmark — Serial vs Multiprocessing

This section measures the real-world speedup of `ProcessPoolExecutor` over serial execution on a CPU-bound task (prime sieve).  
Expected speedup is approximately `cpu_count` on a lightly-loaded machine.

In [33]:
def count_primes_range(args):
    """Count primes in [start, end)."""
    start, end = args
    return sum(1 for n in range(start, end) if is_prime(n))

LIMIT = 200_000
N_CPUS = mp.cpu_count()

# Split work into equal chunks, one per CPU
chunk = LIMIT // N_CPUS
ranges = [(i * chunk, (i + 1) * chunk) for i in range(N_CPUS)]

# Serial
t0 = time.perf_counter()
serial_count = sum(count_primes_range(r) for r in ranges)
serial_t = time.perf_counter() - t0

# Multiprocessing via Pool
t0 = time.perf_counter()
with Pool(N_CPUS) as pool:
    parallel_count = sum(pool.map(count_primes_range, ranges))
parallel_t = time.perf_counter() - t0

print(f'Primes below {LIMIT:,}: {serial_count}')
assert serial_count == parallel_count, 'Counts differ!'

print(f'\nCPUs used:  {N_CPUS}')
print(f'Serial:     {serial_t:.3f}s')
print(f'Parallel:   {parallel_t:.3f}s')
print(f'Speedup:    {serial_t / parallel_t:.2f}x')

Primes below 200,000: 17984

CPUs used:  8
Serial:     0.271s
Parallel:   0.118s
Speedup:    2.30x


## 9. Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Passing a lambda as worker | `PicklingError` | Use a module-level or importable `def` |
| Missing `if __name__ == '__main__'` guard (Windows/spawn) | Recursive process spawning | Wrap entry point in the guard |
| Mutating shared state without a lock | Non-deterministic results | Use `Lock`, `Value`, or `Queue` |
| Forgetting `p.join()` | Zombie processes, premature exit | Always join, or use context manager |
| Large data in Queue | Slow serialization, OOM | Pass file paths or shared memory instead |
| Daemon child spawning children | `AssertionError` | Don't set `daemon=True` if child needs workers |

In [34]:
import pickle

# Pitfall 1: lambdas are not picklable — multiprocessing requires pickling
fn = lambda x: x ** 2
try:
    pickle.dumps(fn)
    print('Lambda pickled (unlikely)')
except Exception as e:
    print(f'Lambda pickle error: {type(e).__name__}: {e}')

# Named functions pickle fine
def square(x): return x ** 2
print('Named function picklable:', pickle.dumps(square) is not None)

Lambda pickle error: PicklingError: Can't pickle <function <lambda> at 0x7acf3de7ec20>: attribute lookup <lambda> on __main__ failed
Named function picklable: True


In [35]:
# Pitfall 2: mutable default / global state is copied, not shared
GLOBAL_LIST = []

def append_to_global(val):
    GLOBAL_LIST.append(val)   # modifies the child's private copy

p = Process(target=append_to_global, args=(42,))
p.start()
p.join()

# GLOBAL_LIST in the main process is unchanged
print('GLOBAL_LIST in main after child ran:', GLOBAL_LIST)
print('Use Queue or shared Value/Array to communicate results back to the parent.')

# Pitfall 3: zombie processes if join() is omitted
def noop(): pass

p = Process(target=noop)
p.start()
# Without p.join() the process is a zombie until the parent exits.
# Always call join() or use p as a context manager in Python 3.3+.
p.join()   # correct
print('Process joined cleanly, exitcode:', p.exitcode)

GLOBAL_LIST in main after child ran: []
Use Queue or shared Value/Array to communicate results back to the parent.
Process joined cleanly, exitcode: 0


In [36]:
# Start method: fork vs spawn vs forkserver
# Linux default: fork  — fast, inherits parent state (risk: inherits locks/file handles)
# macOS/Windows: spawn — safe, but slower; requires workers to be picklable and importable
# forkserver: safe + fast for large apps; uses a dedicated server process

print('Current start method:', mp.get_start_method())

# Switch start method at the top of a script (before any Pool/Process is created):
#   mp.set_start_method('spawn')      # safe cross-platform default
#   mp.set_start_method('forkserver') # better for large processes

# The if __name__ == '__main__' guard is mandatory on Windows/spawn
# because spawn re-imports the module in each child process:
print()
print('Always protect Pool/Process calls with:')
print('  if __name__ == "__main__":')
print('      with Pool() as pool: ...')

Current start method: fork

Always protect Pool/Process calls with:
  if __name__ == "__main__":
      with Pool() as pool: ...


## 10. `asyncio` — Cooperative Concurrency for I/O

`asyncio` runs a single-threaded **event loop** that multiplexes many concurrent tasks by suspending them at `await` points while waiting for I/O.  
There is no parallelism — only one coroutine runs at a time — but I/O wait time is shared, so throughput can be high.

**Core concepts**

| Concept | Syntax | Purpose |
|---------|--------|---------|
| Coroutine | `async def fn()` | A function that can be suspended |
| Await | `await expr` | Suspend until the awaitable resolves |
| Task | `asyncio.create_task(coro)` | Schedule a coroutine to run concurrently |
| Gather | `await asyncio.gather(*coros)` | Run multiple coroutines concurrently, collect results |
| Sleep | `await asyncio.sleep(n)` | Non-blocking sleep (releases event loop) |
| Entry point | `asyncio.run(main())` | Start the event loop; use `await` directly in Jupyter |
| Queue | `asyncio.Queue` | Async-safe FIFO for producer/consumer patterns |
| Semaphore | `asyncio.Semaphore(n)` | Limit concurrent coroutines (e.g., API rate limiting) |

> **Jupyter note:** ipykernel runs its own event loop, so use `await` directly at the top level of a cell instead of `asyncio.run()`.

In [37]:
# Coroutine basics: async def + await
async def fetch(label, delay):
    print(f'[{label}] start')
    await asyncio.sleep(delay)   # releases the event loop while "waiting"
    print(f'[{label}] done after {delay}s')
    return label

# Sequential — total time ≈ sum of delays
t0 = time.perf_counter()
r1 = await fetch('A', 0.1)
r2 = await fetch('B', 0.1)
print(f'Sequential: {time.perf_counter() - t0:.3f}s  results: {r1}, {r2}')

# Concurrent with gather — total time ≈ max of delays
t0 = time.perf_counter()
r1, r2 = await asyncio.gather(fetch('C', 0.1), fetch('D', 0.1))
print(f'Concurrent: {time.perf_counter() - t0:.3f}s  results: {r1}, {r2}')

[A] start
[A] done after 0.1s
[B] start
[B] done after 0.1s
Sequential: 0.201s  results: A, B
[C] start
[D] start
[C] done after 0.1s
[D] done after 0.1s
Concurrent: 0.102s  results: C, D


In [38]:
# create_task schedules a coroutine immediately; await it later
async def slow_job(name, delay):
    await asyncio.sleep(delay)
    return f'{name} result'

# Tasks start running as soon as created — overlap naturally
t0 = time.perf_counter()
task_a = asyncio.create_task(slow_job('alpha', 0.15))
task_b = asyncio.create_task(slow_job('beta',  0.10))
task_c = asyncio.create_task(slow_job('gamma', 0.05))

# Await in any order — they all run concurrently
results = await asyncio.gather(task_a, task_b, task_c)
print(f'Tasks done in {time.perf_counter() - t0:.3f}s: {results}')

# as_completed equivalent for tasks — process results as each finishes
import asyncio

async def timed_job(n):
    await asyncio.sleep(n * 0.05)
    return n ** 2

tasks = [asyncio.create_task(timed_job(i)) for i in range(1, 6)]
done_order = []
for coro in asyncio.as_completed(tasks):
    val = await coro
    done_order.append(val)
print('Completion order:', done_order)

Tasks done in 0.151s: ['alpha result', 'beta result', 'gamma result']
Completion order: [1, 4, 9, 16, 25]


In [39]:
# asyncio.Queue: async-safe producer/consumer
async def producer(q, items):
    for item in items:
        await asyncio.sleep(0.02)   # simulate data arrival
        await q.put(item)
        print(f'produced {item}')
    await q.put(None)               # sentinel to stop the consumer

async def consumer(q):
    results = []
    while True:
        item = await q.get()
        if item is None:
            break
        results.append(item * 10)
        print(f'consumed {item}')
    return results

q = asyncio.Queue(maxsize=3)
_, results = await asyncio.gather(producer(q, [1, 2, 3, 4]), consumer(q))
print('Consumer results:', results)

# asyncio.Semaphore: limit concurrent coroutines (e.g., API rate limiting)
async def rate_limited_fetch(sem, url_id):
    async with sem:
        await asyncio.sleep(0.05)   # simulate HTTP request
        return f'response-{url_id}'

sem = asyncio.Semaphore(2)   # at most 2 concurrent "requests"
t0 = time.perf_counter()
responses = await asyncio.gather(*[rate_limited_fetch(sem, i) for i in range(6)])
print(f'Rate-limited fetch: {time.perf_counter() - t0:.3f}s  ({len(responses)} responses)')

produced 1
consumed 1
produced 2
consumed 2
produced 3
consumed 3
produced 4
consumed 4
Consumer results: [10, 20, 30, 40]
Rate-limited fetch: 0.152s  (6 responses)


## 11. Combining `asyncio` with `multiprocessing`

`asyncio` handles I/O concurrently but is single-threaded — CPU-bound work blocks the event loop.  
`loop.run_in_executor(ProcessPoolExecutor, fn, *args)` offloads CPU work to a process pool and returns an awaitable, keeping the event loop free.

**Pattern:** async coroutine orchestrates I/O tasks *and* CPU tasks simultaneously.

```
event loop (single thread)
  ├── await asyncio.sleep(...)     → I/O overlap, no blocking
  ├── await loop.run_in_executor(ProcessPoolExecutor, cpu_fn, x)
  │       → CPU work runs in a subprocess in parallel
  └── await asyncio.gather(...)    → both I/O and CPU tasks overlap
```

| Need | Approach |
|------|----------|
| CPU-bound from async code | `loop.run_in_executor(ProcessPoolExecutor, fn, arg)` |
| Blocking I/O from async code | `loop.run_in_executor(None, fn)` — uses `ThreadPoolExecutor` |
| Blocking stdlib call | `await asyncio.to_thread(fn, *args)` (Python 3.9+, thread-based) |

In [40]:
def cpu_heavy(n):
    """CPU-bound: sum of square roots — blocks the caller."""
    return sum(math.sqrt(i) for i in range(n))

loop = asyncio.get_event_loop()

# run_in_executor wraps a blocking call and returns an awaitable
with concurrent.futures.ProcessPoolExecutor() as proc_pool:
    t0 = time.perf_counter()
    # submit two CPU tasks to the process pool and one I/O task concurrently
    cpu_task1 = loop.run_in_executor(proc_pool, cpu_heavy, 2_000_000)
    cpu_task2 = loop.run_in_executor(proc_pool, cpu_heavy, 2_000_000)
    io_task   = asyncio.sleep(0.1)   # simulated I/O

    results = await asyncio.gather(cpu_task1, cpu_task2, io_task)
    elapsed = time.perf_counter() - t0

print(f'CPU result 1: {results[0]:.2f}')
print(f'CPU result 2: {results[1]:.2f}')
print(f'Total time (CPU + I/O overlap): {elapsed:.3f}s')

CPU result 1: 1885617375.85
CPU result 2: 1885617375.85
Total time (CPU + I/O overlap): 0.195s


In [41]:
# Practical pipeline: async I/O fetch → CPU processing → async write
# Each stage overlaps with others; the event loop stays responsive throughout.

def process_batch(batch):
    """CPU-bound: compute sum of squares for a batch of numbers."""
    return sum(x ** 2 for x in batch)

async def async_fetch(batch_id):
    """Simulate async I/O — fetches a batch of data."""
    await asyncio.sleep(0.05)
    return list(range(batch_id * 10, (batch_id + 1) * 10))

async def async_save(batch_id, result):
    """Simulate async I/O — writes result to storage."""
    await asyncio.sleep(0.02)
    return f'saved batch {batch_id} -> {result}'

async def pipeline(batch_id, loop, pool):
    batch   = await async_fetch(batch_id)                             # async I/O
    result  = await loop.run_in_executor(pool, process_batch, batch)  # CPU in process
    summary = await async_save(batch_id, result)                      # async I/O
    return summary

loop = asyncio.get_event_loop()

t0 = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor() as pool:
    summaries = await asyncio.gather(*[pipeline(i, loop, pool) for i in range(6)])
elapsed = time.perf_counter() - t0

for s in summaries:
    print(s)
print(f'\nAll 6 batches processed in {elapsed:.3f}s')

saved batch 0 -> 285
saved batch 1 -> 2185
saved batch 2 -> 6085
saved batch 3 -> 11985
saved batch 4 -> 19885
saved batch 5 -> 29785

All 6 batches processed in 0.120s


## 12. Summary

### API decision guide

| Need | Use |
|------|-----|
| Run one function in a separate process | `Process(target=fn, args=(...))` |
| Map a function over a list in parallel | `Pool.map` / `Pool.starmap` |
| Higher-level, Future-based interface | `ProcessPoolExecutor` |
| Stream results as they complete | `as_completed(futures)` |
| Share a single integer/float | `Value('i', 0)` or `Value('d', 0.0)` |
| Share a fixed-size array | `Array('d', size)` |
| Protect shared state | `Lock` (context manager) |
| Rate-limit concurrent access | `Semaphore(n)` |
| Broadcast stop signal | `Event.set()` |
| Fan-out results from workers | `Queue` |
| Concurrent I/O in a single thread | `async def` + `await asyncio.gather(...)` |
| Concurrent tasks with individual control | `asyncio.create_task()` |
| Async-safe producer/consumer | `asyncio.Queue` |
| Limit concurrent coroutines | `asyncio.Semaphore(n)` |
| CPU work from async code | `loop.run_in_executor(ProcessPoolExecutor, fn, arg)` |
| Blocking I/O from async code | `loop.run_in_executor(None, fn)` or `asyncio.to_thread(fn)` |

### Rules of thumb

- Use **multiprocessing** for CPU-bound work; **asyncio** for I/O-bound; **threading** only when asyncio isn't an option.
- Worker functions passed to `ProcessPoolExecutor` must be **picklable** — module-level `def`, never lambdas.
- Always **join** every process you start; use context managers to guarantee cleanup.
- Protect every access to shared `Value`/`Array` with a `Lock`.
- Avoid passing large objects through `Queue`; prefer paths or shared memory.
- On non-Linux platforms, wrap the entry point in `if __name__ == '__main__'`.
- In `asyncio`, never call blocking functions directly — they freeze the event loop; offload to `run_in_executor`.
- Use `asyncio.gather` when tasks are independent; use `asyncio.create_task` when you need handles for cancellation or callbacks.